# Download W&B Training Curves

Download mean return and mean success curves from Weights & Biases for Experiments 1, 2, and 3.

The completed runs did not upload local `episodes.csv`, so this notebook downloads the scalar history that is available online:
`train/mean_return_100` and `train/success_rate_100`. Experiment 3 can be downloaded while it is still running; rerun the notebook later to refresh the CSVs.


## 1. Settings


In [1]:
# ----------------------------------------------------------------------------
# EDIT ME
# ----------------------------------------------------------------------------
WANDB_ENTITY = None          # None -> use $WANDB_ENTITY or the logged-in default entity
WANDB_PROJECT = "ppo-cf"
OUTPUT_DIR = "wandb_training_logs"

# Any subset of: "experiment_1", "experiment_2", "experiment_3".
EXPERIMENTS = ["experiment_1", "experiment_2", "experiment_3"]

# Set to None to keep all run states. Useful while Experiment 3 is still running.
RUN_STATES = None            # e.g. ["finished", "running"]

# Set to False if you want to include older/manual runs that happen to match names loosely.
STRICT_EXPERIMENT_PREFIX = True

METRICS = {
    "mean_return": "train/mean_return_100",
    "mean_success": "train/success_rate_100",
}
# ----------------------------------------------------------------------------


## 2. Setup


In [2]:
import os
import json
import re
from datetime import datetime, timezone
from functools import reduce
from pathlib import Path

import pandas as pd

try:
    import wandb
except ImportError as exc:
    raise RuntimeError("Install wandb in this environment first: pip install wandb") from exc

ROOT = Path.cwd()
OUT = ROOT / OUTPUT_DIR
RAW = OUT / "raw_runs"
for path in (OUT, RAW):
    path.mkdir(parents=True, exist_ok=True)

print("output:", OUT)


output: /Users/charithapalika/Desktop/Personal Projects/PPO_CF/notebooks/wandb_training_logs


## 3. Connect To W&B


In [3]:
api = wandb.Api(timeout=60)

entity = WANDB_ENTITY or os.environ.get("WANDB_ENTITY")
if entity is None:
    viewer = getattr(api, "viewer", None)
    entity = getattr(viewer, "entity", None) or getattr(viewer, "username", None)
if entity is None:
    raise RuntimeError("Could not infer W&B entity. Set WANDB_ENTITY in the first cell.")

PROJECT_PATH = f"{entity}/{WANDB_PROJECT}"
print("project:", PROJECT_PATH)


wandb: Currently logged in as: charithapalika (charitha_palika) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


project: charitha_palika/ppo-cf


## 4. Run Selection Helpers


In [4]:
def safe_filename(text):
    keep = []
    for ch in str(text):
        keep.append(ch if ch.isalnum() or ch in "._-" else "_")
    return "".join(keep).strip("_") or "run"


def tags_of(run):
    return {str(t) for t in (run.tags or [])}


def cfg_get(cfg, dotted, default=None):
    if dotted in cfg:
        return cfg[dotted]
    cur = cfg
    for part in dotted.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return default
        cur = cur[part]
    return cur


def experiment_of(run):
    name = str(run.name or "").lower()
    group = str(run.group or "").lower()
    cfg = dict(run.config or {})
    namespace = str(cfg_get(cfg, "wandb.run_namespace", "")).lower()
    tags = {t.lower() for t in tags_of(run)}

    if name.startswith("exp1_") or group.startswith("exp1_"):
        return "experiment_1"
    if name.startswith("e2_") or group.startswith("e2_") or namespace.startswith("e2_"):
        return "experiment_2"
    if name.startswith("e3_") or group.startswith("e3_") or namespace.startswith("e3_") or any(t.startswith("query-") for t in tags):
        return "experiment_3"

    if STRICT_EXPERIMENT_PREFIX:
        return None

    if "exp1" in name or "exp1" in group:
        return "experiment_1"
    if "e2" in name or "e2" in group:
        return "experiment_2"
    if "e3" in name or "e3" in group:
        return "experiment_3"
    return None


def infer_seed(run):
    cfg = dict(run.config or {})
    seed = cfg.get("seed")
    if seed is not None:
        return seed
    match = re.search(r"seed(\d+)", str(run.name or "").lower())
    return int(match.group(1)) if match else None


def infer_env_tag(run):
    known = ["doorkey5x5", "doorkey6x6", "lavagap", "unlock", "unlockpickup", "redbluedoors6x6", "taxi"]
    tags = {t.lower() for t in tags_of(run)}
    name = str(run.name or "").lower()
    group = str(run.group or "").lower()
    cfg = dict(run.config or {})
    env_id = str(cfg_get(cfg, "env.env_id", "")).lower()
    blob = " ".join([name, group, env_id, " ".join(tags)]).replace("_", "").replace("-", "")
    aliases = {
        "doorkey5x5": ["doorkey5x5", "dk5"],
        "doorkey6x6": ["doorkey6x6", "dk6"],
        "lavagap": ["lavagap"],
        "unlock": ["unlock"],
        "unlockpickup": ["unlockpickup"],
        "redbluedoors6x6": ["redbluedoors6x6", "rbd6"],
        "taxi": ["taxi"],
    }
    # Check longer names first so unlockpickup is not classified as unlock.
    for env in ["unlockpickup", "redbluedoors6x6", "doorkey5x5", "doorkey6x6", "lavagap", "unlock", "taxi"]:
        if env in tags:
            return env
        for alias in aliases[env]:
            if alias.replace("_", "").replace("-", "") in blob:
                return env
    return None


def infer_algo_tag(run):
    tags = {t.lower() for t in tags_of(run)}
    for tag in ["ppo", "ppo-cf", "queried", "distill", "uniform", "uncertainty", "active"]:
        if tag in tags:
            return tag
    job = str(getattr(run, "job_type", "") or "").lower()
    if job:
        return job
    name = str(run.name or "").lower()
    if "active" in name:
        return "active"
    if "uncertainty" in name:
        return "uncertainty"
    if "uniform" in name:
        return "uniform"
    if "distill" in name:
        return "distill"
    if "queried" in name:
        return "queried"
    if "_cf" in name or "ppo-cf" in name:
        return "ppo-cf"
    if "gae" in name or "ppo" in name:
        return "ppo"
    return None


def infer_beta(run):
    cfg = dict(run.config or {})
    beta = cfg_get(cfg, "distill.beta")
    if beta is not None:
        return beta
    for text in [str(run.group or ""), str(run.name or ""), " ".join(tags_of(run))]:
        match = re.search(r"beta-([0-9.]+)", text)
        if match:
            return float(match.group(1))
        match = re.search(r"_b(\d+)p(\d+)", text)
        if match:
            return float(f"{match.group(1)}.{match.group(2)}")
    return None


def infer_query_strategy(run):
    cfg = dict(run.config or {})
    strategy = cfg_get(cfg, "distill.query_strategy")
    if strategy:
        return strategy
    for tag in tags_of(run):
        if str(tag).startswith("query-"):
            return str(tag).replace("query-", "", 1)
    return None


## 5. Select Runs


In [5]:
selected = []
state_filter = set(RUN_STATES) if RUN_STATES is not None else None
experiment_filter = set(EXPERIMENTS)

for run in api.runs(PROJECT_PATH):
    exp = experiment_of(run)
    if exp not in experiment_filter:
        continue
    if state_filter is not None and str(run.state) not in state_filter:
        continue
    selected.append((run, exp))

selected = sorted(selected, key=lambda x: (x[1], str(x[0].group or ""), str(x[0].name or "")))
print("selected runs:", len(selected))
for run, exp in selected[:40]:
    print(f"  {exp:12s} {str(run.name):42s} state={str(run.state):10s} group={str(run.group)}")
if len(selected) > 40:
    print("  ...")


selected runs: 226
  experiment_1 exp1_dk5_cf_seed0                          state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_cf_seed1                          state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_cf_seed2                          state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_cf_seed3                          state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_cf_seed4                          state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_gae_seed0                         state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_gae_seed1                         state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_gae_seed2                         state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_gae_seed3                         state=finished   group=EXP1_DK5
  experiment_1 exp1_dk5_gae_seed4                         state=finished   group=EXP1_DK5
  experiment_1 exp1_dk6_cf_seed0                          state=finished   group=

## 6. Download Metric Histories


In [6]:
def download_metric(run, output_name, wandb_key):
    rows = []
    for row in run.scan_history(keys=["_step", wandb_key], page_size=1000):
        if wandb_key not in row:
            continue
        rows.append({
            "global_step": row.get("_step"),
            output_name: row.get(wandb_key),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["global_step", output_name])
    return df.drop_duplicates(subset=["global_step"], keep="last").sort_values("global_step")


def download_run(run, experiment):
    metric_frames = [download_metric(run, out_name, wandb_key) for out_name, wandb_key in METRICS.items()]
    non_empty = [df for df in metric_frames if not df.empty]
    if non_empty:
        hist = reduce(lambda left, right: left.merge(right, on="global_step", how="outer"), non_empty)
        hist = hist.sort_values("global_step")
    else:
        hist = pd.DataFrame(columns=["global_step", *METRICS.keys()])

    hist.insert(0, "query_strategy", infer_query_strategy(run))
    hist.insert(0, "beta", infer_beta(run))
    hist.insert(0, "seed", infer_seed(run))
    hist.insert(0, "algo_tag", infer_algo_tag(run))
    hist.insert(0, "env_tag", infer_env_tag(run))
    hist.insert(0, "experiment", experiment)
    hist.insert(0, "wandb_state", run.state)
    hist.insert(0, "wandb_group", run.group)
    hist.insert(0, "wandb_run_name", run.name)
    hist.insert(0, "wandb_run_id", run.id)
    return hist

all_history = []
run_rows = []
failed = []

for idx, (run, experiment) in enumerate(selected, 1):
    print(f"[{idx:03d}/{len(selected):03d}] {experiment} {run.name}", flush=True)
    try:
        hist = download_run(run, experiment)
        if len(hist):
            path = RAW / f"{experiment}__{safe_filename(run.name)}__{run.id}.csv"
            hist.to_csv(path, index=False)
            all_history.append(hist)
        cfg = dict(run.config or {})
        run_rows.append({
            "experiment": experiment,
            "wandb_run_id": run.id,
            "wandb_run_name": run.name,
            "wandb_group": run.group,
            "wandb_job_type": getattr(run, "job_type", None),
            "wandb_state": run.state,
            "wandb_url": run.url,
            "env_tag": infer_env_tag(run),
            "algo_tag": infer_algo_tag(run),
            "seed": infer_seed(run),
            "beta": infer_beta(run),
            "query_strategy": infer_query_strategy(run),
            "n_history_rows": int(len(hist)),
            "created_at": str(getattr(run, "created_at", "")),
            "config_env_id": cfg_get(cfg, "env.env_id"),
            "config_pg_mode": cfg_get(cfg, "ppo.pg_mode"),
            "config_run_name": cfg_get(cfg, "run.run_name"),
            "wandb_namespace": cfg_get(cfg, "wandb.run_namespace"),
        })
    except Exception as exc:
        failed.append({
            "experiment": experiment,
            "wandb_run_id": run.id,
            "wandb_run_name": run.name,
            "error": repr(exc),
        })
        print("  FAILED", repr(exc), flush=True)

history = pd.concat(all_history, ignore_index=True) if all_history else pd.DataFrame()
runs_df = pd.DataFrame(run_rows)
failed_df = pd.DataFrame(failed)

history.to_csv(OUT / "training_history.csv", index=False)
runs_df.to_csv(OUT / "runs.csv", index=False)
failed_df.to_csv(OUT / "failed_downloads.csv", index=False)

manifest = {
    "project": PROJECT_PATH,
    "downloaded_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "experiments": EXPERIMENTS,
    "run_states": RUN_STATES,
    "metrics": METRICS,
    "n_selected_runs": len(selected),
    "n_history_rows": int(len(history)),
    "n_failed": len(failed),
    "note": "Downloaded W&B scalar histories. Raw local episodes.csv files were not available in W&B for earlier runs.",
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))

print()
print("wrote:")
for path in [OUT / "training_history.csv", OUT / "runs.csv", OUT / "failed_downloads.csv", OUT / "manifest.json"]:
    print(" ", path.relative_to(ROOT))
print("per-run csvs:", len(list(RAW.glob("*.csv"))))


[001/226] experiment_1 exp1_dk5_cf_seed0
[002/226] experiment_1 exp1_dk5_cf_seed1


KeyboardInterrupt: 

## 7. Final Values And Coverage


In [ ]:
if history.empty:
    raise RuntimeError("No history rows downloaded. Check W&B login/entity/project and run filters.")

sort_cols = ["experiment", "env_tag", "algo_tag", "seed", "beta", "query_strategy", "wandb_run_name", "global_step"]
sort_cols = [c for c in sort_cols if c in history.columns]
final_metrics = (
    history.sort_values(sort_cols)
    .groupby(["experiment", "wandb_run_id"], as_index=False)
    .tail(1)
    .sort_values(["experiment", "env_tag", "algo_tag", "seed", "beta", "query_strategy", "wandb_run_name"], na_position="last")
)
final_metrics.to_csv(OUT / "final_metrics.csv", index=False)
display(final_metrics)
print("wrote", (OUT / "final_metrics.csv").relative_to(ROOT))

coverage_cols = ["experiment", "env_tag", "algo_tag", "beta", "query_strategy"]
coverage_cols = [c for c in coverage_cols if c in runs_df.columns]
coverage = (
    runs_df.groupby(coverage_cols, dropna=False)
    .agg(n_runs=("wandb_run_id", "nunique"), n_rows=("n_history_rows", "sum"))
    .reset_index()
    .sort_values(coverage_cols, na_position="last")
)
coverage.to_csv(OUT / "coverage.csv", index=False)
display(coverage)
print("wrote", (OUT / "coverage.csv").relative_to(ROOT))


## 8. Load Existing Download Later


In [ ]:
history_path = OUT / "training_history.csv"
runs_path = OUT / "runs.csv"
if history_path.exists() and runs_path.exists():
    history = pd.read_csv(history_path)
    runs_df = pd.read_csv(runs_path)
    display(history.head())
    display(runs_df.head())
else:
    print("No saved download found yet.")
